# Oral–Gut Microbiome Fusion — Replication Notebook

**Companion to:** Ruthbah, C.A., Sadi, T.H., Jahan, N.E.S., & Adib, A.N.M.T. (2026).  
*Interpretable Machine Learning Reveals Complementary Age-Related Signatures in the Oral and Gut Microbiome.* bioRxiv. https://doi.org/10.64898/2026.09.09.750358


**What this notebook does:**  
1. Loads and preprocesses the **Ferretti et al. (2018)** cohort (n=44, adult vs. newborn) — the primary cohort from the paper.  
2. Trains classifiers on stool-only, oral-only, and fused conditions; evaluates cross-validated AUC.  
3. Runs SHAP site-attribution on the fused model to quantify each body site's contribution.  
4. Loads and preprocesses the **Brito et al. (2016)** cohort (n=140, female vs. male) — the independent replication cohort.  
5. Repeats the same pipeline on Brito and compares site-attribution splits across both cohorts.  
6. Runs a permutation significance test on both cohorts and produces the comparison figure (Figure 5 of the paper).

**Prerequisites:**  
- Run `fetch_ferretti.R` and `fetch_brito.R` first to produce the four CSV files this notebook reads.  
- All CSV files are expected at `/content/` (Google Colab) — adjust paths for local use.  
- Required Python packages: `numpy`, `pandas`, `scikit-learn`, `xgboost`, `shap`, `matplotlib`.

**Runtime:** ~5 min (Ferretti) + ~12 min (Brito) on a standard CPU.

In [ ]:
# ── Imports and output directory ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("/content/results", exist_ok=True)

# ── Load Ferretti et al. (2018) CSVs ─────────────────────────────────────────
# These files are produced by fetch_ferretti.R from curatedMetagenomicData.
# abundance_raw.csv: rows = samples, columns = taxa (relative abundances).
# metadata_raw.csv: rows = samples, columns include subject_id, body_site, age_category.
abundance = pd.read_csv("/content/abundance_raw.csv")
meta      = pd.read_csv("/content/metadata_raw.csv")

# Column name constants — change these only if your metadata uses different column names.
SUBJECT_COL   = "subject_id"
SITE_COL      = "body_site"
LABEL_COL     = "age_category"
SAMPLE_ID_COL = "sample_id"
SITES_TO_USE  = ["stool", "oralcavity"]

# ── Filter to subjects with samples at BOTH sites ─────────────────────────────
# We keep only one sample per subject per site (the first chronologically).
# This avoids pseudo-replication in subjects sampled multiple times.
meta_filtered = meta[meta[SITE_COL].isin(SITES_TO_USE)].copy()
meta_filtered = meta_filtered.sort_values(SAMPLE_ID_COL).drop_duplicates(
    subset=[SUBJECT_COL, SITE_COL], keep="first"
)
site_counts       = meta_filtered.groupby(SUBJECT_COL)[SITE_COL].nunique()
complete_subjects = site_counts[site_counts == len(SITES_TO_USE)].index.tolist()
meta_complete     = meta_filtered[meta_filtered[SUBJECT_COL].isin(complete_subjects)].copy()

print(f"Ferretti: {len(complete_subjects)} subjects with matched stool + oral samples")
print(meta_complete.groupby(SUBJECT_COL)[LABEL_COL].first().value_counts())

## Shared helper functions (used for both cohorts)

In [ ]:
# ── build_site_table() ───────────────────────────────────────────────────────
# Extracts the feature matrix and labels for one body site from the metadata
# and abundance tables. Returns (features_df, labels_series) indexed by subject_id.
def build_site_table(site_name, meta_df, abundance_df):
    site_meta = meta_df[meta_df[SITE_COL] == site_name][[SUBJECT_COL, SAMPLE_ID_COL, LABEL_COL]]
    merged    = site_meta.merge(abundance_df, on=SAMPLE_ID_COL, how="left")
    merged    = merged.set_index(SUBJECT_COL)
    labels    = merged[LABEL_COL]
    features  = merged.drop(columns=[SAMPLE_ID_COL, LABEL_COL])
    features  = features.select_dtypes(include=[np.number])
    return features, labels


# ── clr_transform() ──────────────────────────────────────────────────────────
# Centered log-ratio (CLR) transformation for compositional microbiome data.
# WHY: Raw relative abundances are constrained to sum to 1.0, which creates
# spurious negative correlations between taxa (if one goes up, others must go
# down, mathematically). CLR breaks this constraint by expressing each taxon's
# abundance relative to the geometric mean of all taxa in that sample.
# A small pseudocount is added before the log to handle zero abundances
# (many taxa are absent in most samples).
def clr_transform(df, pseudocount=1e-6):
    df     = df + pseudocount          # avoid log(0)
    log_df = np.log(df)
    gm     = log_df.mean(axis=1)      # per-sample geometric mean (in log space)
    return log_df.sub(gm, axis=0)     # subtract geometric mean from each taxon


# ── filter_low_variance() ────────────────────────────────────────────────────
# Removes taxa that are nearly constant across subjects.
# WHY: A taxon present at ~0.001% in every subject carries no discriminative
# signal for classification. Including it adds noise and inflates feature count.
# Default threshold of 1e-4 removes taxa that barely vary — adjust downward
# (e.g. 1e-5) if too many features are removed.
def filter_low_variance(df, min_variance=1e-4):
    variances = df.var(axis=0)
    return df[variances[variances > min_variance].index]


# ── get_site_contribution() ──────────────────────────────────────────────────
# Trains a Random Forest on the fused feature matrix, computes SHAP values,
# and sums mean absolute SHAP importance by body site of origin.
# WHY SHAP: Accuracy metrics can't distinguish a redundant second site from a
# complementary one that's simply not needed to reach the accuracy ceiling.
# SHAP reveals how much each site actually influenced the model's predictions,
# regardless of whether removing it would lower accuracy.
def get_site_contribution(conditions_dict, y_binary_series, sites_list, fused_name):
    import shap
    X_fused = conditions_dict[fused_name].fillna(0)

    # Train Random Forest on the full fused matrix (not cross-validated here —
    # we need the fitted model to compute SHAP values on the same data).
    model = RandomForestClassifier(n_estimators=300, random_state=42)
    model.fit(X_fused, y_binary_series)

    # TreeExplainer is the fast, exact SHAP method for tree-based models.
    explainer = shap.TreeExplainer(model)
    raw_shap  = explainer.shap_values(X_fused)

    # shap_values() output format varies by shap version:
    # - older versions: list [class_0_array, class_1_array]
    # - newer versions: single 3D array (subjects × features × classes)
    # We always extract the class-1 (positive class) SHAP matrix.
    if isinstance(raw_shap, np.ndarray) and raw_shap.ndim == 3:
        shap_matrix = raw_shap[:, :, 1]
    elif isinstance(raw_shap, list):
        shap_matrix = np.array(raw_shap[1])
    elif hasattr(raw_shap, "values"):
        vals = raw_shap.values
        shap_matrix = vals[:, :, 1] if vals.ndim == 3 else vals
    else:
        shap_matrix = np.array(raw_shap)

    # Mean absolute SHAP value per feature across all subjects.
    # Absolute value because we care about magnitude of influence, not direction.
    mean_abs_shap = np.abs(shap_matrix).mean(axis=0)

    # Sum SHAP importance for all features belonging to each site.
    # Feature names are prefixed with their site (e.g. "stool__Bacteroides_vulgatus").
    feature_names   = X_fused.columns
    site_importance = {}
    for site in sites_list:
        prefix = f"{site}__"
        idx    = [i for i, f in enumerate(feature_names) if f.startswith(prefix)]
        site_importance[site] = mean_abs_shap[idx].sum() if idx else 0

    return site_importance, shap_matrix, X_fused

## Part A — Ferretti et al. (2018): Primary cohort

In [ ]:
# Install xgboost and shap if not already present.
# The -q flag suppresses verbose output; remove it if you want to see install logs.
!pip install xgboost shap -q

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import shap
import warnings
warnings.filterwarnings("ignore")

# ── Three classifiers ─────────────────────────────────────────────────────────
# LogReg_L1:    L1-penalised logistic regression — sparse, interpretable baseline.
#               StandardScaler is required because logistic regression is sensitive
#               to feature scale; CLR-transformed data still varies in magnitude.
# RandomForest: Ensemble of 300 decision trees — robust, no scaling needed,
#               compatible with TreeExplainer for exact SHAP values.
# XGBoost:      Gradient-boosted trees — stronger learner, also TreeExplainer-compatible.
models = {
    "LogReg_L1":    Pipeline([("scale", StandardScaler()),
                               ("clf",   LogisticRegression(penalty="l1", solver="liblinear",
                                                             max_iter=2000, random_state=42))]),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
    "XGBoost":      XGBClassifier(n_estimators=200, max_depth=3,
                                   use_label_encoder=False, eval_metric="logloss",
                                   random_state=42)
}

In [ ]:
# ── Build per-site feature tables for Ferretti ───────────────────────────────
site_tables = {}
site_labels = {}
for site in SITES_TO_USE:
    feats, labs = build_site_table(site, meta_complete, abundance)
    site_tables[site] = feats
    site_labels[site] = labs
    print(f"{site}: {feats.shape[0]} subjects, {feats.shape[1]} taxa (before filtering)")

# ── Preprocess: variance filter → CLR transform ───────────────────────────────
# Order matters: filter first (on raw abundances), then CLR-transform.
# Filtering after CLR would use transformed variances, which are less intuitive.
site_tables_clr = {}
for site in SITES_TO_USE:
    filtered              = filter_low_variance(site_tables[site])
    site_tables_clr[site] = clr_transform(filtered)
    print(f"{site}: {filtered.shape[1]} taxa retained after variance filtering")

# ── Align subjects across sites ───────────────────────────────────────────────
# Keep only subjects present in ALL sites (intersection).
# This ensures each subject contributes exactly one row to every site table.
common_subjects = set(site_tables_clr[SITES_TO_USE[0]].index)
for site in SITES_TO_USE[1:]:
    common_subjects &= set(site_tables_clr[site].index)
common_subjects = sorted(common_subjects)

for site in SITES_TO_USE:
    site_tables_clr[site] = site_tables_clr[site].loc[common_subjects]

# ── Binary labels: 1 = adult, 0 = newborn ─────────────────────────────────────
y        = site_labels[SITES_TO_USE[0]].loc[common_subjects]
y_binary = (y.astype(str).str.lower() == "adult").astype(int)
print(f"\nLabel balance — adult: {y_binary.sum()}, newborn: {(y_binary==0).sum()}")

# ── Three classification conditions ──────────────────────────────────────────
# stool-only, oral-only, and early fusion (column names prefixed by site).
# Prefixing is critical: it lets SHAP site-attribution identify which features
# came from which body site after concatenation.
conditions = {}
for site in SITES_TO_USE:
    conditions[site] = site_tables_clr[site]
all_name         = "+".join(SITES_TO_USE)
conditions[all_name] = pd.concat(
    [site_tables_clr[s].add_prefix(f"{s}__") for s in SITES_TO_USE], axis=1
)

print(f"\nFerretti final aligned subjects: {len(common_subjects)}")
for name, df in conditions.items():
    print(f"  {name}: {df.shape}")

In [ ]:
# ── 5-fold stratified cross-validation for all conditions × all models ────────
# Stratified: each fold preserves the adult/newborn class ratio.
# AUC (area under ROC curve) is used because it is threshold-independent and
# appropriate for binary classification regardless of class imbalance.
cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
for cond_name, X in conditions.items():
    X_clean = X.fillna(0)   # replace any NaN with 0 (rare, but safe)
    for model_name, model in models.items():
        scores = cross_val_score(model, X_clean, y_binary, cv=cv, scoring="roc_auc")
        results.append({
            "condition": cond_name,
            "model":     model_name,
            "mean_auc":  scores.mean(),
            "std_auc":   scores.std()
        })

results_df = pd.DataFrame(results)
print("Ferretti cross-validated AUC results:")
print(results_df.to_string(index=False))
# Expected: stool Random Forest AUC = 1.000 ± 0.000
# Fusion does not exceed stool alone — this is the accuracy ceiling the paper discusses.

### A1 — SHAP site-attribution on the Ferretti fused model

In [ ]:
# ── SHAP site-attribution: Ferretti fused model ──────────────────────────────
# Even though stool alone reaches AUC=1.00, we ask: inside the fused model,
# how much does each site actually contribute to the predictions?
# If the model relied only on stool features, oral SHAP importance would be ~0.
# The result below shows it is not — oral cavity contributes ~58% despite being
# "unnecessary" for accuracy. This is the paper's central methodological finding.
ferretti_site_importance, ferretti_shap_matrix, ferretti_X_fused = get_site_contribution(
    conditions, y_binary, SITES_TO_USE, "+".join(SITES_TO_USE)
)
total = sum(ferretti_site_importance.values())
print("Ferretti (n=44) SHAP site contribution:")
for site, val in ferretti_site_importance.items():
    print(f"  {site}: {val/total*100:.1f}%")
# Expected: stool ~41.9%, oralcavity ~58.1%

## Part B — Brito et al. (2016): Independent replication cohort

**Why a second cohort?**  
The Ferretti cohort has only 44 subjects. A directional finding at n=44 could be a statistical accident.  
Brito et al. (2016) provides 140 matched stool + oral subjects from a completely different population,  
with a harder classification task (sex instead of age group), making it an independent and more demanding test.

**Data source:** `fetch_brito.R` fetches `BritoIL_2016.relative_abundance` from curatedMetagenomicData  
and writes two CSV files: `brito_abundance_raw.csv` and `brito_metadata_raw.csv`.  
Run that script before executing the cells below.

In [ ]:
# ── Load Brito et al. (2016) CSVs ────────────────────────────────────────────
# These are produced by fetch_brito.R (not fetch_nagata.R — that was a naming
# error in an earlier version of this notebook, now corrected).
brito_abundance = pd.read_csv("/content/brito_abundance_raw.csv")
brito_meta      = pd.read_csv("/content/brito_metadata_raw.csv")

print("Brito metadata columns:", list(brito_meta.columns))

# ── Inspect available body sites ──────────────────────────────────────────────
site_col_guess = [c for c in brito_meta.columns if "body" in c.lower() or "site" in c.lower()]
print("\nLikely body-site column(s):", site_col_guess)
if site_col_guess:
    print(brito_meta[site_col_guess[0]].value_counts())

# ── Inspect potential classification targets ──────────────────────────────────
# We chose 'gender' (sex) as the target because:
#   - age_category: all subjects are adults, so no contrast
#   - disease: all subjects are healthy (no disease label)
#   - gender: balanced male/female — the most meaningful available binary contrast
print("\nAvailable classification target columns:")
for col in ["disease", "study_condition", "age_category", "gender", "BMI", "disease_subtype"]:
    if col in brito_meta.columns:
        print(f"\n{col}:")
        print(brito_meta[col].value_counts(dropna=False).head(10))

In [ ]:
# ── Brito cohort column configuration ────────────────────────────────────────
# Using the same column names as Ferretti — curatedMetagenomicData standardises these.
BRITO_SUBJECT_COL   = "subject_id"
BRITO_SITE_COL      = "body_site"
BRITO_LABEL_COL     = "gender"      # sex used as classification target
BRITO_SAMPLE_ID_COL = "sample_id"
BRITO_SITES_TO_USE  = ["stool", "oralcavity"]
BRITO_TARGET_VALUE  = "female"      # positive class (1 = female, 0 = male)

# ── Filter to subjects with matched stool + oral samples ─────────────────────
brito_meta_filtered = brito_meta[brito_meta[BRITO_SITE_COL].isin(BRITO_SITES_TO_USE)].copy()
brito_meta_filtered = brito_meta_filtered.sort_values(BRITO_SAMPLE_ID_COL).drop_duplicates(
    subset=[BRITO_SUBJECT_COL, BRITO_SITE_COL], keep="first"
)
brito_site_counts       = brito_meta_filtered.groupby(BRITO_SUBJECT_COL)[BRITO_SITE_COL].nunique()
brito_complete_subjects = brito_site_counts[brito_site_counts == 2].index.tolist()
brito_meta_complete     = brito_meta_filtered[
    brito_meta_filtered[BRITO_SUBJECT_COL].isin(brito_complete_subjects)
].copy()

print(f"Brito: {len(brito_complete_subjects)} subjects with matched stool + oral samples")
print(brito_meta_complete.groupby(BRITO_SUBJECT_COL)[BRITO_LABEL_COL].first().value_counts(dropna=False))

### B1 — Build feature tables, preprocess, fuse, and classify (Brito)

In [ ]:
# ── Build per-site feature tables for Brito ──────────────────────────────────
# Reuse the same build_site_table(), clr_transform(), filter_low_variance()
# functions defined earlier — the pipeline is identical to Ferretti.
brito_site_tables = {}
brito_site_labels = {}
for site in BRITO_SITES_TO_USE:
    feats = brito_meta_complete[
        brito_meta_complete[BRITO_SITE_COL] == site
    ][[BRITO_SUBJECT_COL, BRITO_SAMPLE_ID_COL]].merge(
        brito_abundance, on=BRITO_SAMPLE_ID_COL, how="left"
    ).set_index(BRITO_SUBJECT_COL)

    labs = brito_meta_complete[
        brito_meta_complete[BRITO_SITE_COL] == site
    ].set_index(BRITO_SUBJECT_COL)[BRITO_LABEL_COL]

    feats = feats.drop(columns=[BRITO_SAMPLE_ID_COL], errors="ignore")
    feats = feats.select_dtypes(include=[np.number])
    brito_site_tables[site] = feats
    brito_site_labels[site] = labs
    print(f"{site}: {feats.shape[0]} subjects, {feats.shape[1]} taxa (before filtering)")

# ── Preprocess: variance filter → CLR transform (per site) ───────────────────
brito_site_tables_clr = {}
for site in BRITO_SITES_TO_USE:
    filtered                   = filter_low_variance(brito_site_tables[site])
    brito_site_tables_clr[site] = clr_transform(filtered)
    print(f"{site}: {filtered.shape[1]} taxa retained after variance filtering")

# ── Align subjects (intersection across both sites) ───────────────────────────
brito_common_subjects = set(brito_site_tables_clr[BRITO_SITES_TO_USE[0]].index)
for site in BRITO_SITES_TO_USE[1:]:
    brito_common_subjects &= set(brito_site_tables_clr[site].index)
brito_common_subjects = sorted(brito_common_subjects)

for site in BRITO_SITES_TO_USE:
    brito_site_tables_clr[site] = brito_site_tables_clr[site].loc[brito_common_subjects]

# ── Binary labels: 1 = female, 0 = male ──────────────────────────────────────
brito_y_raw    = brito_site_labels[BRITO_SITES_TO_USE[0]].loc[brito_common_subjects]
brito_y_binary = (brito_y_raw == BRITO_TARGET_VALUE).astype(int)
print(f"\nLabel balance — female (1): {brito_y_binary.sum()}, male (0): {(brito_y_binary==0).sum()}")

# ── Three conditions: stool-only, oral-only, fused ────────────────────────────
brito_conditions = {}
for site in BRITO_SITES_TO_USE:
    brito_conditions[site] = brito_site_tables_clr[site]
brito_fused_name = "+".join(BRITO_SITES_TO_USE)
brito_conditions[brito_fused_name] = pd.concat(
    [brito_site_tables_clr[s].add_prefix(f"{s}__") for s in BRITO_SITES_TO_USE], axis=1
)

print(f"\nBrito final aligned subjects: {len(brito_common_subjects)}")
for name, df in brito_conditions.items():
    print(f"  {name}: {df.shape}")

In [ ]:
# ── Cross-validated AUC for Brito cohort ─────────────────────────────────────
# Sex-based microbiome classification is a genuinely hard task (AUC ~0.55–0.65
# is expected and biologically meaningful — not a pipeline failure).
# This harder task is used deliberately to test whether the SHAP site-attribution
# pattern holds even when no accuracy ceiling is reached.
brito_cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
brito_results = []
for cond_name, X in brito_conditions.items():
    X_clean = X.fillna(0)
    for model_name, model in models.items():
        try:
            scores = cross_val_score(
                model, X_clean, brito_y_binary, cv=brito_cv, scoring="roc_auc"
            )
            brito_results.append({
                "condition": cond_name,
                "model":     model_name,
                "mean_auc":  scores.mean(),
                "std_auc":   scores.std()
            })
        except Exception as e:
            print(f"SKIPPED {cond_name}/{model_name}: {e}")

brito_results_df = pd.DataFrame(brito_results)
print("Brito cross-validated AUC results:")
print(brito_results_df.to_string(index=False))
brito_results_df.to_csv("/content/results/brito_condition_scores.csv", index=False)
# Expected: all AUC values modest (0.53–0.63). Fusion again does not exceed best single site.

### B2 — SHAP site-attribution on the Brito fused model

In [ ]:
# ── SHAP site-attribution: Brito fused model ─────────────────────────────────
# Same get_site_contribution() function used for Ferretti.
# Despite a completely different dataset and classification task, we expect to
# see oral cavity dominating SHAP importance at a similar ratio (~60/40).
brito_site_importance, brito_shap_matrix, brito_X_fused = get_site_contribution(
    brito_conditions, brito_y_binary, BRITO_SITES_TO_USE, brito_fused_name
)
brito_total = sum(brito_site_importance.values())
print(f"Brito (n={len(brito_common_subjects)}) SHAP site contribution:")
for site, val in brito_site_importance.items():
    print(f"  {site}: {val/brito_total*100:.1f}%")
# Expected: stool ~39.3%, oralcavity ~60.7%

# ── Bar chart: Brito site contribution ───────────────────────────────────────
plt.figure(figsize=(6, 4))
pd.Series({k: v/brito_total*100 for k, v in brito_site_importance.items()}).plot(
    kind="bar", color="darkorange", edgecolor="black"
)
plt.axhline(50, color="gray", linestyle="--", linewidth=1, label="50% (equal contribution)")
plt.ylabel("% of total mean |SHAP value|")
plt.title(f"Brito et al. (n={len(brito_common_subjects)}): SHAP site contribution")
plt.xticks(rotation=0)
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig("/content/results/brito_site_contribution.png", dpi=150)
plt.show()

## Part C — Permutation significance test (both cohorts)

**What this test asks:**  
Is the observed ~60/40 oral/stool SHAP split meaningfully different from what you'd get  
if you randomly relabelled which features belonged to which site?

**How it works:**  
1. Take the already-computed mean absolute SHAP values per feature (from the fused model).  
2. Randomly shuffle which features are "assigned" to stool vs. oral (preserving site sizes).  
3. Recompute the site-contribution split under this random assignment.  
4. Repeat 200 times to build a null distribution of max-site shares.  
5. p-value = fraction of null permutations ≥ observed max-site share.

**Interpretation:**  
- Low p (< 0.05): the asymmetry is unlikely under random feature assignment — real site-level structure.  
- High p: the asymmetry could arise by chance at this sample size (expected at n=44; see paper).

In [ ]:
def site_split_permutation_test(X_fused_df, y_binary_series, sites_list,
                                real_importance, n_permutations=200, seed=42):
    """
    Tests whether the observed SHAP site-contribution asymmetry is statistically
    distinguishable from random feature-to-site assignment.

    Parameters
    ----------
    X_fused_df       : pd.DataFrame  — fused feature matrix (subjects × features),
                        column names prefixed by site (e.g. 'stool__taxon_A').
    y_binary_series  : array-like    — binary classification labels.
    sites_list       : list of str   — site names (must match column prefixes).
    real_importance  : dict          — observed SHAP importance per site
                        (from get_site_contribution()).
    n_permutations   : int           — number of random reassignments (200 for
                        exploratory; 1000+ recommended for publication).
    seed             : int           — random seed for reproducibility.

    Returns
    -------
    dict with: real_split_pct, real_max_pct, null_mean_max_pct,
               null_std_max_pct, p_value, null_distribution (np.ndarray).
    """
    import shap
    rng           = np.random.RandomState(seed)
    feature_names = list(X_fused_df.columns)

    # Count how many features belong to each site in the real (observed) split.
    # The permutation preserves these counts — it only scrambles which specific
    # columns are grouped together, not how many are in each group.
    site_sizes = {
        site: sum(1 for f in feature_names if f.startswith(f"{site}__"))
        for site in sites_list
    }

    # Re-fit the model and recompute SHAP values (ensures consistency with
    # the real_importance values passed in).
    model = RandomForestClassifier(n_estimators=300, random_state=42)
    model.fit(X_fused_df, y_binary_series)
    explainer = shap.TreeExplainer(model)
    raw_shap  = explainer.shap_values(X_fused_df)

    if isinstance(raw_shap, np.ndarray) and raw_shap.ndim == 3:
        shap_matrix = raw_shap[:, :, 1]
    elif isinstance(raw_shap, list):
        shap_matrix = np.array(raw_shap[1])
    elif hasattr(raw_shap, "values"):
        vals = raw_shap.values
        shap_matrix = vals[:, :, 1] if vals.ndim == 3 else vals
    else:
        shap_matrix = np.array(raw_shap)

    mean_abs_shap = np.abs(shap_matrix).mean(axis=0)

    # Observed (real) max site share
    real_total   = sum(real_importance.values())
    real_pct     = {s: real_importance[s] / real_total * 100 for s in sites_list}
    real_max_pct = max(real_pct.values())

    # ── Permutation loop ───────────────────────────────────────────────────────
    null_max_pcts = []
    all_idx       = np.arange(len(feature_names))
    for _ in range(n_permutations):
        shuffled_idx  = rng.permutation(all_idx)
        pos           = 0
        perm_importance = {}
        for site in sites_list:
            size                   = site_sizes[site]
            idx_for_site           = shuffled_idx[pos:pos + size]
            perm_importance[site]  = mean_abs_shap[idx_for_site].sum()
            pos                   += size
        perm_total = sum(perm_importance.values())
        perm_pct   = {s: perm_importance[s] / perm_total * 100 for s in sites_list}
        null_max_pcts.append(max(perm_pct.values()))

    null_max_pcts = np.array(null_max_pcts)
    # p-value: proportion of null permutations with max-site share >= observed
    p_value = (null_max_pcts >= real_max_pct).mean()

    return {
        "real_split_pct":    real_pct,
        "real_max_pct":      real_max_pct,
        "null_mean_max_pct": null_max_pcts.mean(),
        "null_std_max_pct":  null_max_pcts.std(),
        "p_value":           p_value,
        "null_distribution": null_max_pcts
    }

In [ ]:
# ── Ferretti permutation test ─────────────────────────────────────────────────
# Expected: p ~0.89 — the asymmetry is directionally consistent but not
# statistically significant at n=44. This reflects limited power, not absence
# of the effect (confirmed by the Brito result below).
print("Running permutation test on Ferretti site-contribution split (200 permutations)...")
ferretti_perm_result = site_split_permutation_test(
    ferretti_X_fused, y_binary, SITES_TO_USE, ferretti_site_importance, n_permutations=200
)
print(f"  Observed split:   {ferretti_perm_result['real_split_pct']}")
print(f"  Observed max:     {ferretti_perm_result['real_max_pct']:.1f}%")
print(f"  Null (mean±std):  {ferretti_perm_result['null_mean_max_pct']:.1f}% ± {ferretti_perm_result['null_std_max_pct']:.1f}%")
print(f"  P-value:          {ferretti_perm_result['p_value']:.4f}")

print()

# ── Brito permutation test ────────────────────────────────────────────────────
# Expected: p < 0.005 — at n=140 the asymmetry is statistically significant.
# The observed max-site share exceeds all 200 permuted null values.
print("Running permutation test on Brito site-contribution split (200 permutations)...")
brito_perm_result = site_split_permutation_test(
    brito_X_fused, brito_y_binary, BRITO_SITES_TO_USE, brito_site_importance, n_permutations=200
)
print(f"  Observed split:   {brito_perm_result['real_split_pct']}")
print(f"  Observed max:     {brito_perm_result['real_max_pct']:.1f}%")
print(f"  Null (mean±std):  {brito_perm_result['null_mean_max_pct']:.1f}% ± {brito_perm_result['null_std_max_pct']:.1f}%")
print(f"  P-value:          {brito_perm_result['p_value']:.4f}")

## Part D — Cross-cohort comparison figure (Figure 5 of the paper)

In [ ]:
# ── Side-by-side permutation null distributions ───────────────────────────────
# Left panel: Ferretti (n=44) — p=0.89, not significant (limited power)
# Right panel: Brito (n=140)  — p<0.005, statistically significant
# Both panels show the same directional pattern (~58–61% oral dominance).
# The consistency across two independent cohorts is the key replication finding.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, perm_result, label, n in [
    (axes[0], ferretti_perm_result, "Ferretti", len(common_subjects)),
    (axes[1], brito_perm_result,    "Brito",    len(brito_common_subjects))
]:
    ax.hist(perm_result["null_distribution"], bins=20,
            color="gray", edgecolor="black", alpha=0.7,
            label="Null (random assignment)")
    ax.axvline(perm_result["real_max_pct"], color="red", linestyle="--",
               linewidth=2, label=f"Observed ({perm_result['real_max_pct']:.1f}%)")
    ax.axvline(50, color="blue", linestyle=":",
               linewidth=1.5, label="50% (no asymmetry)")

    p_str = (f"p = {perm_result['p_value']:.4f}"
             if perm_result["p_value"] > 0 else "p < 0.005")
    ax.set_title(f"{label} (n={n})\n{p_str}", fontsize=11)
    ax.set_xlabel("Max site share of SHAP importance (%)", fontsize=10)
    ax.set_ylabel("Count", fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle("Permutation null distributions — SHAP site-attribution", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("/content/results/permutation_test_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nInterpretation:")
print("  LOW p-value (< 0.05): observed site-contribution asymmetry is unlikely under")
print("  random feature-to-site reassignment — indicates real, site-level structure.")
print("  Both cohorts show the same ~60/40 oral/stool split, despite different tasks")
print("  and sample sizes — this cross-cohort consistency is the replication finding.")

## Part E — Save results summary

In [ ]:
# ── Export a machine-readable summary of the key numbers ─────────────────────
# This JSON file records the exact figures reported in the paper,
# making it easy to verify reproducibility without re-reading plots.
import json, shutil

summary = {
    "ferretti": {
        "n_subjects":            len(common_subjects),
        "classification_target": "adult_vs_newborn",
        "site_contribution_pct": {k: round(v/total*100, 2)
                                    for k, v in ferretti_site_importance.items()},
        "permutation_p_value":   float(ferretti_perm_result["p_value"])
    },
    "brito": {
        "n_subjects":            len(brito_common_subjects),
        "classification_target": "sex_female_vs_male",
        "positive_class":        BRITO_TARGET_VALUE,
        "site_contribution_pct": {k: round(v/brito_total*100, 2)
                                    for k, v in brito_site_importance.items()},
        "permutation_p_value":   float(brito_perm_result["p_value"])
    }
}

with open("/content/results/replication_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

# ── Zip all outputs into a single downloadable archive ────────────────────────
# Contents: brito_condition_scores.csv, brito_site_contribution.png,
#           permutation_test_comparison.png, replication_summary.json
shutil.make_archive("/content/replication_results", "zip", "/content/results")
print("\nAll outputs saved to: /content/replication_results.zip")